In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure Matplotlib and Seaborn style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)

# Load cleaned dataset from File 2
data_path = os.path.join('data', 'all_months_clean.csv')

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f" Cleaned dataset loaded successfully. Shape: {df.shape}")
else:
    raise FileNotFoundError(f"File not found at '{data_path}'. Run 02_Data_Cleaning_and_Standardization first.")

In [ ]:
# Separate numerical, categorical, and district columns
price_cols = ['Min_Price', 'Max_Price', 'Avg_Price']
supply_cols = ['Volume', 'TOTAL_sources', 'reconciliation_gap', 'import_share']
meta_cols = ['Product_Name', 'Category', 'month_name', 'month_idx', 'bs_year', 'bs_month']

district_cols = [c for c in df.columns if c not in price_cols + supply_cols + meta_cols + ['Unit', 'unit_canonical', 'unit_changed', 'Total_Amount', 'Volume_Equals', 'n_months_present', 'is_balanced']]

print("=== SUMMARY STATISTICS FOR PRICES ===")
display(df[price_cols].describe())

print("\n=== SUMMARY STATISTICS FOR SUPPLY VOLUMES ===")
display(df[supply_cols].describe())

print("\n=== MISSING VALUES PROFILE ===")
print(df[price_cols + supply_cols].isnull().sum())

In [ ]:
# Aggregate average price and total volume by Product
product_summary = df.groupby('Product_Name').agg(
    Avg_Price=('Avg_Price', 'mean'),
    Total_Volume=('Volume', 'sum'),
    Import_Share=('import_share', 'mean'),
    Category=('Category', 'first'),
    Months_Present=('month_idx', 'nunique')
).reset_index()

print(" Top 10 Most Supplied Products by Volume:")
display(product_summary.sort_values(by='Total_Volume', ascending=False).head(10))

print("\n Top 10 Highest Price Products (Avg NPR):")
display(product_summary.sort_values(by='Avg_Price', ascending=False).head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price distribution (Log scale due to skewness)
sns.histplot(df['Avg_Price'].dropna(), kde=True, ax=axes[0], color='teal', bins=30)
axes[0].set_title('Average Price Distribution (NPR)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Avg Price (NPR)')
axes[0].set_ylabel('Frequency')

# Volume distribution
sns.histplot(df['Volume'].dropna(), kde=True, ax=axes[1], color='coral', bins=30)
axes[1].set_title('Supply Volume Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Volume')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

# Plot mean Min, Avg, and Max prices across chronological months
monthly_price = df.groupby('month_idx')[['Min_Price', 'Avg_Price', 'Max_Price']].mean()

plt.plot(monthly_price.index, monthly_price['Min_Price'], label='Min Price (Avg)', marker='o', linestyle='--', color='green')
plt.plot(monthly_price.index, monthly_price['Avg_Price'], label='Avg Price', marker='s', linewidth=2.5, color='blue')
plt.plot(monthly_price.index, monthly_price['Max_Price'], label='Max Price (Avg)', marker='^', linestyle='--', color='red')

plt.title('Monthly Price Band Progression (BS Chronological Months 1–10)', fontsize=13, fontweight='bold')
plt.xlabel('Chronological Month Index (1 = Shrawan 2082, 10 = Baishakh 2083)')
plt.ylabel('Price (NPR)')
plt.xticks(range(1, 11))
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

# Category average prices
plt.subplot(1, 2, 1)
sns.boxplot(data=df, x='Category', y='Avg_Price', palette='Set2')
plt.title('Price Distribution by Category', fontweight='bold')
plt.ylabel('Avg Price (NPR)')

# Category total volume
plt.subplot(1, 2, 2)
df.groupby('Category')['Volume'].sum().plot(kind='bar', color=['skyblue', 'lightgreen', 'orange'])
plt.title('Total Supply Volume by Category', fontweight='bold')
plt.ylabel('Total Volume')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Identify top 15 supply source districts/origins overall
top_districts = df[district_cols].sum().sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 5))
sns.barplot(x=top_districts.values, y=top_districts.index, palette='viridis')
plt.title('Top 15 Supply Sources by Total Volume', fontsize=13, fontweight='bold')
plt.xlabel('Aggregated Supply Volume')
plt.ylabel('District / Source Origin')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

# Plot import share trend over time
monthly_import = df.groupby('month_idx')['import_share'].mean() * 100

plt.plot(monthly_import.index, monthly_import.values, marker='o', color='crimson', linewidth=2)
plt.title('Average Monthly Import Share (%)', fontsize=13, fontweight='bold')
plt.xlabel('Chronological Month Index (1–10)')
plt.ylabel('Import Share (%)')
plt.xticks(range(1, 11))
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

corr_features = ['Avg_Price', 'Min_Price', 'Max_Price', 'Volume', 'TOTAL_sources', 'import_share', 'reconciliation_gap']
corr_matrix = df[corr_features].corr()

sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("=== BALANCED PANEL AUDIT SUMMARY ===")
print(f"Total Unique Products: {df['Product_Name'].nunique()}")
print(f"Balanced Products (Present in all 10 Months): {df[df['is_balanced']]['Product_Name'].nunique()}")
print(f"Unbalanced Products (<10 Months): {df[~df['is_balanced']]['Product_Name'].nunique()}")

print("\n Unbalanced Products Breakdown:")
display(df[~df['is_balanced']].groupby('Product_Name')['month_idx'].nunique().reset_index().rename(columns={'month_idx': 'Months_Recorded'}))